# 2026 season exploration

A first look at the 2026 data collected by `src/collect_data.py`, ahead of building features for the US GP win-probability model. Run `.venv/bin/python src/collect_data.py` first.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from fastf1.ergast import Ergast

from src import config  # enables the FastF1 cache

pd.set_option('display.width', 120)
race = pd.read_csv(config.DATA_DIR / '2026_race_results.csv')
quali = pd.read_csv(config.DATA_DIR / '2026_qualifying_results.csv')
LAST_ROUND = race['Round'].max()
print(f'Rounds loaded: 1-{LAST_ROUND}, {race.shape[0]} race rows, {quali.shape[0]} qualifying rows')

Rounds loaded: 1-17, 374 race rows, 369 qualifying rows


## Championship standings after round 17

Six 2026 rounds were sprint weekends (China, Miami, Canada, Britain, Netherlands, Singapore), and sprint points count toward the championship. Our race-results CSV only has Grand Prix points, so the official standings come from the Jolpica (Ergast-compatible) API via FastF1.

In [2]:
ergast = Ergast()
drivers = ergast.get_driver_standings(2026, round=LAST_ROUND).content[0]
drivers['team'] = drivers['constructorNames'].str.join(' / ')
driver_standings = drivers[['position', 'driverCode', 'givenName', 'familyName', 'team', 'points', 'wins']]
driver_standings.set_index('position')

,driverCode,givenName,familyName,team,points,wins
position,,,,,,
1,ANT,Andrea Kimi,Antonelli,Mercedes,340.0,8
2,RUS,George,Russell,Mercedes,238.0,3
3,HAM,Lewis,Hamilton,Ferrari,229.0,1
4,VER,Max,Verstappen,Red Bull,221.0,2
5,LEC,Charles,Leclerc,Ferrari,215.0,1
6,NOR,Lando,Norris,McLaren,201.0,2
7,PIA,Oscar,Piastri,McLaren,138.0,0
8,HAD,Isack,Hadjar,Red Bull,97.0,0
9,LAW,Liam,Lawson,RB F1 Team / Red Bull,75.0,0


In [3]:
constructors = ergast.get_constructor_standings(2026, round=LAST_ROUND).content[0]
constructors[['position', 'constructorName', 'points', 'wins']].set_index('position')

,constructorName,points,wins
position,,,
1,Mercedes,578.0,11
2,Ferrari,444.0,2
3,McLaren,339.0,2
4,Red Bull,332.0,2
5,RB F1 Team,100.0,0
6,Alpine F1 Team,68.0,0
7,Haas F1 Team,30.0,0
8,Audi,23.0,0
9,Williams,12.0,0


Lawson appears with two teams (RB F1 Team / Red Bull) because he changed seats mid-season, which is why the standings list 23 drivers for 22 seats.

For reference, the gap between official points and Grand Prix-only points shows how much the sprints contributed:

In [4]:
gp_points = race.groupby('Abbreviation')['Points'].sum()
cmp = driver_standings.set_index('driverCode')[['points']].rename(columns={'points': 'official'})
cmp['gp_only'] = gp_points.reindex(cmp.index).fillna(0)
cmp['sprint_points'] = cmp['official'] - cmp['gp_only']
cmp.head(10)

,official,gp_only,sprint_points
driverCode,,,
ANT,340.0,309.0,31.0
RUS,238.0,204.0,34.0
HAM,229.0,202.0,27.0
VER,221.0,201.0,20.0
LEC,215.0,181.0,34.0
NOR,201.0,168.0,33.0
PIA,138.0,117.0,21.0
HAD,97.0,97.0,0.0
LAW,75.0,68.0,7.0


## Winners by round

In [5]:
winners = race[race['Position'] == 1][['Round', 'EventName', 'Abbreviation', 'TeamName', 'GridPosition']]
winners = winners.sort_values('Round').rename(columns={'Abbreviation': 'Winner', 'GridPosition': 'Grid'})
winners['Grid'] = winners['Grid'].astype(int)
winners.set_index('Round')

,EventName,Winner,TeamName,Grid
Round,,,,
1,Australian Grand Prix,RUS,Mercedes,1
2,Chinese Grand Prix,ANT,Mercedes,1
3,Japanese Grand Prix,ANT,Mercedes,1
4,Miami Grand Prix,ANT,Mercedes,1
5,Canadian Grand Prix,ANT,Mercedes,2
6,Monaco Grand Prix,ANT,Mercedes,1
7,Barcelona Grand Prix,HAM,Ferrari,2
8,Austrian Grand Prix,RUS,Mercedes,1
9,British Grand Prix,LEC,Ferrari,2


In [6]:
winners['TeamName'].value_counts().rename('wins')

TeamName
Mercedes    11
Ferrari      2
McLaren      2
Red Bull     2
Name: wins, dtype: int64

## How often did the pole sitter win?

Pole = P1 in qualifying. Grid P1 can differ if the pole sitter took a grid penalty, so both are shown.

In [7]:
pole = quali[quali['Position'] == 1][['Round', 'Abbreviation']].rename(columns={'Abbreviation': 'Pole'})
grid1 = race[race['GridPosition'] == 1][['Round', 'Abbreviation']].rename(columns={'Abbreviation': 'Grid1'})
pw = winners[['Round', 'EventName', 'Winner']].merge(pole, on='Round').merge(grid1, on='Round', how='left')
pw['pole_won'] = pw['Pole'] == pw['Winner']
n, k = len(pw), pw['pole_won'].sum()
print(f'Pole sitter won {k} of {n} races ({k / n:.0%})')
print(f"Grid P1 won {(pw['Grid1'] == pw['Winner']).sum()} of {n} races")
pw.set_index('Round')

Pole sitter won 12 of 17 races (71%)
Grid P1 won 12 of 17 races


,EventName,Winner,Pole,Grid1,pole_won
Round,,,,,
1,Australian Grand Prix,RUS,RUS,RUS,True
2,Chinese Grand Prix,ANT,ANT,ANT,True
3,Japanese Grand Prix,ANT,ANT,ANT,True
4,Miami Grand Prix,ANT,ANT,ANT,True
5,Canadian Grand Prix,ANT,RUS,RUS,False
6,Monaco Grand Prix,ANT,ANT,ANT,True
7,Barcelona Grand Prix,HAM,RUS,RUS,False
8,Austrian Grand Prix,RUS,RUS,RUS,True
9,British Grand Prix,LEC,ANT,ANT,False


## Average finish over the last 5 races

`Position` is FastF1's finishing order, which places retirements and non-starters behind classified finishers, so a DNF counts as a poor finish rather than being dropped. DNFs are also counted separately.

Drivers are grouped by driver *and* team, so mid-season seat swaps (Lawson, Hadjar, Tsunoda) show up as separate rows.

In [8]:
last5 = race[race['Round'] > LAST_ROUND - 5].copy()
print('Rounds:', ', '.join(last5.drop_duplicates('Round')['EventName'].str.replace(' Grand Prix', '')))
# Not classified (R/D/W/N in ClassifiedPosition) counts as a DNF, even if Status says 'Lapped'
last5['dnf'] = ~last5['ClassifiedPosition'].astype(str).str.isdigit()
driver_form = (last5.groupby(['Abbreviation', 'TeamName'])
               .agg(races=('Position', 'size'), avg_finish=('Position', 'mean'),
                    best=('Position', 'min'), dnfs=('dnf', 'sum'), points=('Points', 'sum'))
               .sort_values('avg_finish').round(2))
driver_form

Rounds: Italian, Spanish, Azerbaijan, Bahrain, Singapore


,,races,avg_finish,best,dnfs,points
Abbreviation,TeamName,,,,,
VER,Red Bull,5,1.80,1.0,0,101.0
ANT,Mercedes,5,2.40,1.0,0,93.0
HAD,Red Bull,3,6.00,3.0,0,26.0
LEC,Ferrari,5,7.20,2.0,1,54.0
RUS,Mercedes,5,7.40,1.0,0,55.0
PIA,McLaren,5,7.40,5.0,0,32.0
NOR,McLaren,5,7.80,3.0,1,41.0
HAM,Ferrari,5,8.60,3.0,1,39.0
LAW,RB F1 Team,3,8.67,7.0,0,12.0


In [9]:
team_form = (last5.groupby('TeamName')
             .agg(avg_finish=('Position', 'mean'), best=('Position', 'min'),
                  dnfs=('dnf', 'sum'), points=('Points', 'sum'))
             .sort_values('avg_finish').round(2))
team_form

,avg_finish,best,dnfs,points
TeamName,,,,
Red Bull,4.7,1.0,0,135.0
Mercedes,4.9,1.0,0,148.0
McLaren,7.6,3.0,1,73.0
Ferrari,7.9,2.0,2,93.0
RB F1 Team,9.9,7.0,0,26.0
Haas F1 Team,12.9,8.0,0,6.0
Alpine F1 Team,13.1,7.0,3,14.0
Audi,13.1,8.0,1,5.0
Williams,16.4,10.0,3,1.0


## Drivers missing from the qualifying file

Every round has 22 race entries, but qualifying has 19 in Australia and 20 in Spain. These are the five race starters with no qualifying row:

In [10]:
m = race.merge(quali[['Round', 'Abbreviation']], on=['Round', 'Abbreviation'], how='left', indicator=True)
missing = m[m['_merge'] == 'left_only'][['Round', 'EventName', 'Abbreviation', 'TeamName', 'GridPosition',
                                          'Position', 'ClassifiedPosition', 'Status']]
missing

,Round,EventName,Abbreviation,TeamName,GridPosition,Position,ClassifiedPosition,Status
5,1,Australian Grand Prix,VER,Red Bull,20.0,6.0,6,Finished
14,1,Australian Grand Prix,SAI,Williams,21.0,15.0,15,Lapped
16,1,Australian Grand Prix,STR,Aston Martin,22.0,17.0,R,Lapped
301,14,Spanish Grand Prix,BEA,Haas F1 Team,22.0,16.0,16,Lapped
306,14,Spanish Grand Prix,STR,Aston Martin,21.0,21.0,R,Retired


All five started from the back of the grid (P20–P22), consistent with not setting a qualifying time (e.g. a crash or mechanical problem in qualifying), so FastF1's qualifying classification has no row for them. They are not race non-starters: Verstappen went from P20 to **P6** in Australia.

**How to handle them in feature building:**

- **Qualifying-pace features** (gap to pole %, quali position): treat these as *missing*, not as slowest. Exclude the session from that driver's weighted quali-pace average. Imputing "last place" would wrongly drag down Verstappen's 2026 pace, when his race result shows the car was quick.
- **Grid-position features**: use the actual `GridPosition` from the race file, which is correct for these rows and is what the race model really needs as a starting-position input.
- **Race-result features**: keep these rows. They are valid race starts.
- **Flag**: add a `no_quali_time` indicator so a recurring pattern (e.g. Stroll, twice) can feed a small reliability/risk term instead of being silently lost.
- **Joins**: always build the driver-round table from the race file and left-join qualifying onto it, so these drivers are never dropped by an inner join.